In [1]:
import time
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import Perceptron, LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


In [2]:
# Use the exact filename from UCI (note: no extension)
df = pd.read_csv("SMSSpamCollection", sep='\t', header=None, names=['label', 'message'])

print(df.shape)
# Preview the data
df.head()


(5572, 2)


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   object
 1   message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [4]:
df["target"] = df["label"].map({"ham": 0, "spam": 1})

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["message"], df["target"], test_size=0.2, random_state=42, stratify=df["target"]
)

In [5]:
# 2. FEATURE EXTRACTION METHODS (2 Vectorizers)
vectorizers = {
    "CountVectorizer (BoW)": CountVectorizer(max_features=5000, stop_words='english', dtype=np.float64),
    "TF-IDF Vectorizer": TfidfVectorizer(max_features=5000, stop_words='english', dtype=np.float64)
}


In [6]:
# ---------------------------------------------------------------------------
# Algorithms
def get_models():
    return {
        "Perceptron": Perceptron(random_state=42),
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
        "Linear SVM": LinearSVC(random_state=42),
        "KNN": KNeighborsClassifier(n_neighbors=5),
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
        "XGBoost": XGBClassifier(eval_metric="logloss", random_state=42),
        "LightGBM": LGBMClassifier(random_state=42, verbose=-1),
        "Neural Network (MLP)": MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=42),
    }


In [7]:
# Train every combination (Nested Loops)
# ---------------------------------------------------------------------------
all_results = []
trained_pipelines = {}

# OUTER LOOP: Iterates through each vectorizer method
for vec_name, vectorizer in vectorizers.items():
    
    # Fit and transform the text data using the current vectorizer
    X_train_vec = vectorizer.fit_transform(X_train_text)
    X_test_vec = vectorizer.transform(X_test_text)
    
    # INNER LOOP: Iterates through each machine learning model
    # Note: We call get_models() here so each vectorizer gets fresh, untrained models
    for model_name, model in get_models().items():
        key = f"{model_name} | {vec_name}"
        try:
            start = time.time()
            model.fit(X_train_vec, y_train)
            train_time = time.time() - start

            preds = model.predict(X_test_vec)

            # Metrics handle 0/1 automatically now (removed pos_label='spam')
            acc = accuracy_score(y_test, preds)
            f1 = f1_score(y_test, preds)  
            precision = precision_score(y_test, preds)
            recall = recall_score(y_test, preds)

            trained_pipelines[key] = {
                "model": model,
                "vectorizer": vectorizer, 
                "vectorizer_name": vec_name,
                "algorithm": model_name,
            }

            all_results.append({
                "Pipeline Key": key,
                "Algorithm": model_name,
                "Vectorizer": vec_name,
                "Accuracy": round(acc, 4),
                "F1 Score": round(f1, 4),
                "Precision": round(precision, 4),
                "Recall": round(recall, 4),
                "Train Time (s)": round(train_time, 2),
            })

            print(f"OK  {key}  (Acc={acc:.4f}, F1={f1:.4f})")

        except Exception as e:
            print(f"FAIL  {key}  -> {e}")

# Process and sort all results after both loops finish running
if all_results:
    results_df = (
        pd.DataFrame(all_results)
        .sort_values(by="F1 Score", ascending=False)
        .reset_index(drop=True)
    )

    best_pipeline_key = results_df.iloc[0]["Pipeline Key"]

    print(f"\n{len(trained_pipelines)} pipelines trained successfully.")
    print(f"Best pipeline: {best_pipeline_key}\n")
    print(results_df.head(12).to_string(index=False)) # Increased head to 12 to show all 2x6 combinations
else:
    print("\nNo pipelines were trained successfully. Check the error messages above.")


OK  Perceptron | CountVectorizer (BoW)  (Acc=0.9722, F1=0.8977)
OK  Logistic Regression | CountVectorizer (BoW)  (Acc=0.9749, F1=0.8971)
OK  Linear SVM | CountVectorizer (BoW)  (Acc=0.9839, F1=0.9362)
OK  KNN | CountVectorizer (BoW)  (Acc=0.9148, F1=0.5320)
OK  Decision Tree | CountVectorizer (BoW)  (Acc=0.9632, F1=0.8541)
OK  Random Forest | CountVectorizer (BoW)  (Acc=0.9767, F1=0.9044)
OK  XGBoost | CountVectorizer (BoW)  (Acc=0.9704, F1=0.8809)
OK  LightGBM | CountVectorizer (BoW)  (Acc=0.9740, F1=0.8997)
OK  Neural Network (MLP) | CountVectorizer (BoW)  (Acc=0.9812, F1=0.9242)
OK  Perceptron | TF-IDF Vectorizer  (Acc=0.9794, F1=0.9210)
OK  Logistic Regression | TF-IDF Vectorizer  (Acc=0.9695, F1=0.8712)
OK  Linear SVM | TF-IDF Vectorizer  (Acc=0.9812, F1=0.9247)
OK  KNN | TF-IDF Vectorizer  (Acc=0.9139, F1=0.5248)
OK  Decision Tree | TF-IDF Vectorizer  (Acc=0.9578, F1=0.8362)
OK  Random Forest | TF-IDF Vectorizer  (Acc=0.9758, F1=0.9004)
OK  XGBoost | TF-IDF Vectorizer  (Acc=0.965

In [11]:
# ---------------------------------------------------------------------------
# Save everything a UI would need
# ---------------------------------------------------------------------------
bundle = {
    "trained_pipelines": trained_pipelines,
    "results_df": results_df,
    "best_pipeline_key": best_pipeline_key,
    "label_map": {0: "ham", 1: "spam"},
}

joblib.dump(bundle, "classification_bundle.pkl")
print("\nSaved classification_bundle.pkl")


Saved classification_bundle.pkl
